In [1]:
!pip -q install timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 43.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 70.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.

In [2]:
import os, random, math
from pathlib import Path
import numpy as np
import cv2
import torch
import torchvision as tv
from torchvision.transforms import functional as TF
import timm
import pandas as pd
from tqdm.auto import tqdm

class CFG:
    # ---- Adjusted for your dataset structure ----
    VOC_ROOT = "/kaggle/input/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val"
    IMG_DIR  = f"{VOC_ROOT}/JPEGImages"
    GT_DIR   = f"{VOC_ROOT}/SegmentationClass"

    OUT_ROOT = "/kaggle/working/outputs/vitroll_all_exp"

    # Model configs
    SEG_BACKBONE = "deeplabv3_resnet50"
    IMG_SIZE = 512
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    VIT_NAME = "deit_base_patch16_224"
    VIT_IN_SZ = 224

    BASE_THRESH = 0.50
    DICE_SWEEP = [0.40, 0.45, 0.50, 0.55, 0.60]

    ALPHA = 1.0
    BETA = 1.0
    ROLLOUT_LAYER_FRACTION = 0.5  # last half of ViT layers

    PERTURBS = ("clean", "blur", "brightness", "gauss", "hflip", "rotation")

def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(42)

os.makedirs(CFG.OUT_ROOT, exist_ok=True)
print("Device:", CFG.DEVICE, "| Out:", CFG.OUT_ROOT)

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Device: cuda | Out: /kaggle/working/outputs/vitroll_all_exp


In [3]:
VAL_TXT = Path(CFG.VOC_ROOT) / "ImageSets" / "Segmentation" / "val.txt"
if VAL_TXT.exists():
    with open(VAL_TXT) as f:
        VAL_IDS = set(x.strip() for x in f if x.strip())
else:
    VAL_IDS = None

print("Using val split:", VAL_TXT.exists(), "| #val ids:", len(VAL_IDS) if VAL_IDS else "ALL")

Using val split: True | #val ids: 1449


In [4]:
def discover_pairs(img_dir, gt_dir, ids=None):
    img_dir, gt_dir = Path(img_dir), Path(gt_dir)
    pairs = []
    if ids is None:
        for p in sorted(img_dir.glob("*.jpg")):
            m = gt_dir / f"{p.stem}.png"
            if m.exists():
                pairs.append((p, m))
    else:
        for stem in sorted(ids):
            p = img_dir / f"{stem}.jpg"
            m = gt_dir  / f"{stem}.png"
            if p.exists() and m.exists():
                pairs.append((p, m))
    print(f"Found {len(pairs)} image/mask pairs")
    return pairs

PAIRS = discover_pairs(CFG.IMG_DIR, CFG.GT_DIR, ids=VAL_IDS)
assert len(PAIRS) > 0, "No pairs found — check dataset path."

Found 1449 image/mask pairs


In [5]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_COLOR)
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

def load_mask(path):
    m = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if m is None: raise FileNotFoundError(path)
    if m.ndim == 3: m = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)
    return (m > 0).astype(np.uint8)

def letterbox(img, size):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = int(round(h*s)), int(round(w*s))
    r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top=(size-nh)//2; bottom=size-nh-top
    left=(size-nw)//2; right=size-nw-left
    out=cv2.copyMakeBorder(r, top,bottom,left,right, cv2.BORDER_CONSTANT, value=(0,0,0))
    return out, dict(pad=(top,bottom,left,right), orig=(h,w))

def unletterbox(x, meta, nearest=True):
    top,bottom,left,right = meta["pad"]
    h0,w0 = meta["orig"]
    x = x[top:x.shape[0]-bottom, left:x.shape[1]-right]
    return cv2.resize(x, (w0,h0), interpolation=cv2.INTER_NEAREST if nearest else cv2.INTER_LINEAR)

def apply_perturb(img, kind):
    if kind=="clean": return img
    if kind=="blur": return cv2.GaussianBlur(img,(5,5),0)
    if kind=="brightness": return np.clip(img.astype(np.float32)*1.25,0,255).astype(np.uint8)
    if kind=="gauss":
        noise = np.random.normal(0,8,img.shape).astype(np.float32)
        return np.clip(img.astype(np.float32)+noise,0,255).astype(np.uint8)
    if kind=="hflip": return np.ascontiguousarray(img[:, ::-1, :])
    if kind=="rotation":
        h,w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2,h//2), 5, 1.0)
        return cv2.warpAffine(img, M, (w,h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    raise ValueError(kind)

In [6]:
def get_seg_model():
    if CFG.SEG_BACKBONE == "fcn_resnet50":
        m = tv.models.segmentation.fcn_resnet50(weights="DEFAULT")
    else:
        m = tv.models.segmentation.deeplabv3_resnet50(weights="DEFAULT")
    return m.eval().to(CFG.DEVICE)

seg_model = get_seg_model()

@torch.inference_mode()
def infer_prob_fg(img_rgb):
    img_r, meta = letterbox(img_rgb, CFG.IMG_SIZE)
    x = torch.from_numpy(img_r).permute(2,0,1).float()/255.0
    x = TF.normalize(x, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]).unsqueeze(0).to(CFG.DEVICE)
    out = seg_model(x)["out"]                 # N x C x H x W
    probs = torch.softmax(out, dim=1)
    p_fg = (1.0 - probs[:,0:1]).squeeze().detach().cpu().numpy()  # foreground prob = 1 - P(bg)
    pad = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE), dtype=np.float32)
    pad[:p_fg.shape[0], :p_fg.shape[1]] = p_fg
    return np.clip(unletterbox(pad, meta, nearest=False), 0, 1).astype(np.float32)

Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth
100%|██████████| 161M/161M [00:00<00:00, 187MB/s]  


In [7]:
import torch
# Use math path, disable flash/mem-efficient SDPA globally
try:
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
except Exception as e:
    print("SDPA toggles not available:", e)

def force_unfused_attn(vit_model):
    for i, blk in enumerate(getattr(vit_model, "blocks", [])):
        attn = getattr(blk, "attn", None)
        if attn is None: continue
        for attr in ("fused_attn", "use_sdpa", "enable_flash_attn"):
            if hasattr(attn, attr):
                setattr(attn, attr, False)

print("Global SDPA disabled. Recreate the ViT model next (Cell 6) and call force_unfused_attn(vit).")

Global SDPA disabled. Recreate the ViT model next (Cell 6) and call force_unfused_attn(vit).


In [8]:
vit = timm.create_model(CFG.VIT_NAME, pretrained=True)
vit.eval().to(CFG.DEVICE)
force_unfused_attn(vit)  

def vit_preprocess(img_rgb):
    im = cv2.resize(img_rgb, (CFG.VIT_IN_SZ, CFG.VIT_IN_SZ), interpolation=cv2.INTER_LINEAR)
    x = torch.from_numpy(im).permute(2,0,1).float()/255.0
    x = TF.normalize(x, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    return x.unsqueeze(0).to(CFG.DEVICE)

def vit_attention_rollout(model, x, layer_fraction=0.5, head_reduce="mean"):
    attns, hooks = [], []
    def hook_fn(mod, inp, out): attns.append(out.detach())  # [B, heads, tokens, tokens]
    # timm ViTs expose blocks[].attn.attn_drop after softmax
    for b in getattr(model, 'blocks', []):
        if hasattr(b, 'attn') and hasattr(b.attn, 'attn_drop'):
            hooks.append(b.attn.attn_drop.register_forward_hook(hook_fn))
    with torch.no_grad(): _ = model(x)
    for h in hooks: h.remove()
    assert len(attns) > 0, "No ViT attention maps captured."

    L = len(attns); K = max(1, int(round(L*layer_fraction)))
    mats = []
    for A in attns[-K:]:
        A = A.mean(dim=1) if head_reduce=="mean" else A.max(dim=1).values  # [B,T,T]
        A = A / (A.sum(dim=-1, keepdim=True) + 1e-6)
        I = torch.eye(A.size(-1), device=A.device).unsqueeze(0)
        A = (A + I); A = A / (A.sum(dim=-1, keepdim=True) + 1e-6)
        mats.append(A)

    R = mats[0]
    for i in range(1,len(mats)): R = R @ mats[i]
    B,T,_ = R.shape; S = int((T-1)**0.5)
    spatial = R[:,1:,0].reshape(B,S,S)
    spatial = (spatial - spatial.min()) / (spatial.max()-spatial.min() + 1e-6)
    spatial = torch.nn.functional.interpolate(spatial.unsqueeze(1),
                                              size=(CFG.VIT_IN_SZ, CFG.VIT_IN_SZ),
                                              mode='bilinear', align_corners=False).squeeze(1)
    return spatial[0].detach().cpu().numpy()  # HxW in [0,1]

def fused_prob_with_rollout(img_rgb, alpha=CFG.ALPHA, beta=CFG.BETA):
    p_fg = infer_prob_fg(img_rgb)
    roll = vit_attention_rollout(vit, vit_preprocess(img_rgb),
                                 layer_fraction=CFG.ROLLOUT_LAYER_FRACTION, head_reduce="mean")
    roll_rs = cv2.resize(roll, (img_rgb.shape[1], img_rgb.shape[0]), interpolation=cv2.INTER_LINEAR)
    fused = (p_fg ** alpha) * (np.clip(roll_rs,0,1) ** beta)
    mn, mx = fused.min(), fused.max()
    if mx > mn: fused = (fused - mn) / (mx - mn)   # keep threshold meaning stable
    return fused.astype(np.float32)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [9]:
# Sanity check: run rollout on one image
img = load_image(PAIRS[0][0])
r = vit_attention_rollout(vit, vit_preprocess(img),
                          layer_fraction=CFG.ROLLOUT_LAYER_FRACTION, head_reduce="mean")
print("rollout shape:", r.shape, "min/max:", float(r.min()), float(r.max()))

rollout shape: (224, 224) min/max: 0.0021821889095008373 0.9830169081687927


In [10]:
def prob_to_mask(prob, thr): return (prob >= thr).astype(np.uint8)

def metrics_iou(pred, gt):
    inter_fg = np.logical_and(pred==1, gt==1).sum()
    union_fg = np.logical_or (pred==1, gt==1).sum()
    iou_fg = inter_fg / (union_fg + 1e-9)
    inter_bg = np.logical_and(pred==0, gt==0).sum()
    union_bg = np.logical_or (pred==0, gt==0).sum()
    iou_bg = inter_bg / (union_bg + 1e-9)
    return float(iou_bg), float(iou_fg), float((iou_bg+iou_fg)/2.0)

def tta_rotate_prob_fused(img_rgb):
    p0 = fused_prob_with_rollout(img_rgb)
    img_h = np.ascontiguousarray(img_rgb[:, ::-1, :]); p_h = fused_prob_with_rollout(img_h)[:, ::-1]
    h,w = img_rgb.shape[:2]
    def rprob(a):
        M = cv2.getRotationMatrix2D((w//2,h//2), a, 1.0)
        imr = cv2.warpAffine(img_rgb, M, (w,h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        return fused_prob_with_rollout(imr)
    p_rp = rprob(+5); p_rn = rprob(-5)
    return np.clip((p0 + p_h + p_rp + p_rn)/4.0, 0, 1)

def morph_from_prob(prob, op="dilate", k=3, thr=CFG.BASE_THRESH):
    mask = prob_to_mask(prob, thr)
    kernel = np.ones((k,k), np.uint8)
    if op == "dilate": out = cv2.dilate(mask, kernel, iterations=1)
    elif op == "erode": out = cv2.erode(mask, kernel, iterations=1)
    else: raise ValueError(op)
    return out

In [11]:
def run_experiment(exp_name, make_mask_fn, out_root="/kaggle/working/outputs/vitroll_all_exp"):
    exp_dir = Path(out_root) / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    for kind in CFG.PERTURBS:
        kind_dir = exp_dir / kind
        (kind_dir/"masks").mkdir(parents=True, exist_ok=True)

        i_b, i_f, m_s = [], [], []
        for img_fp, gt_fp in tqdm(PAIRS, desc=f"{exp_name}:{kind}", leave=False):
            img = load_image(img_fp); gt = load_mask(gt_fp)
            img_p = apply_perturb(img, kind)
            gt_eval = np.ascontiguousarray(gt[:, ::-1]) if kind=="hflip" else gt

            mask = make_mask_fn(img_p)

            ib, ifg, m = metrics_iou(mask, gt_eval)
            i_b.append(ib); i_f.append(ifg); m_s.append(m)

            cv2.imwrite(str(kind_dir/"masks"/f"{img_fp.stem}.png"), (mask*255).astype(np.uint8))

        rows.append({"perturb": kind,
                     "IoU_bg": float(np.mean(i_b)),
                     "IoU_fg": float(np.mean(i_f)),
                     "mIoU":   float(np.mean(m_s))})

    df = pd.DataFrame(rows)[["perturb","IoU_bg","IoU_fg","mIoU"]]
    df.to_csv(exp_dir/"voc_val_robustness.csv", index=False)
    print("Saved:", exp_dir/"voc_val_robustness.csv")
    return df

In [12]:
# G: baseline vitroll
def make_mask_G_vitroll(img_rgb):
    p = fused_prob_with_rollout(img_rgb)
    return prob_to_mask(p, CFG.BASE_THRESH)

# H: rotate/hflip TTA on fused prob
def make_mask_H_vitroll_rotate(img_rgb):
    p = tta_rotate_prob_fused(img_rgb)
    return prob_to_mask(p, CFG.BASE_THRESH)

# I: Dice-calibrated threshold on CLEAN with fusion
_cal_thr_vit = None
def calibrate_threshold_vit_on_clean(maxN=200):
    global _cal_thr_vit
    sample = PAIRS if len(PAIRS) <= maxN else random.sample(PAIRS, maxN)
    scores = []
    for thr in CFG.DICE_SWEEP:
        vals = []
        for img_fp, gt_fp in sample:
            img = load_image(img_fp); gt = load_mask(gt_fp)
            p = fused_prob_with_rollout(img)
            m = prob_to_mask(p, thr)
            tp = np.logical_and(m==1, gt==1).sum()
            fp = np.logical_and(m==1, gt==0).sum()
            fn = np.logical_and(m==0, gt==1).sum()
            dice = (2*tp) / (2*tp + fp + fn + 1e-9)
            vals.append(dice)
        scores.append((thr, float(np.mean(vals))))
    _cal_thr_vit = max(scores, key=lambda x:x[1])[0]
    print("Calibrated vitroll threshold (Dice/Clean):", _cal_thr_vit)

def make_mask_I_vitroll_dicethr(img_rgb):
    global _cal_thr_vit
    if _cal_thr_vit is None: calibrate_threshold_vit_on_clean()
    p = fused_prob_with_rollout(img_rgb)
    return prob_to_mask(p, _cal_thr_vit)

# J: dilate3 after fusion
def make_mask_J_vitroll_dilate3(img_rgb):
    p = fused_prob_with_rollout(img_rgb)
    return morph_from_prob(p, op="dilate", k=3, thr=CFG.BASE_THRESH)

# K: dilate5 after fusion
def make_mask_K_vitroll_dilate5(img_rgb):
    p = fused_prob_with_rollout(img_rgb)
    return morph_from_prob(p, op="dilate", k=5, thr=CFG.BASE_THRESH)

# L: erode3 after fusion
def make_mask_L_vitroll_erode3(img_rgb):
    p = fused_prob_with_rollout(img_rgb)
    return morph_from_prob(p, op="erode", k=3, thr=CFG.BASE_THRESH)

In [ ]:
ALL_EXPS = [
    ("G_vitroll",          make_mask_G_vitroll),
    ("H_vitroll_rotate",   make_mask_H_vitroll_rotate),
    ("I_vitroll_dicethr",  make_mask_I_vitroll_dicethr),
    ("J_vitroll_dilate3",  make_mask_J_vitroll_dilate3),
    ("K_vitroll_dilate5",  make_mask_K_vitroll_dilate5),
    ("L_vitroll_erode3",   make_mask_L_vitroll_erode3),
]


In [13]:
ALL_EXPS = [
    ("L_vitroll_erode3",   make_mask_L_vitroll_erode3),
]


In [14]:
dfs = []
for name, fn in ALL_EXPS:
    df = run_experiment(name, fn, out_root=CFG.OUT_ROOT)
    df["Exp"] = name
    dfs.append(df)

full = pd.concat(dfs, ignore_index=True)

# Pivot to three tables
miou_tbl = full.pivot(index="Exp", columns="perturb", values="mIoU")
bg_tbl   = full.pivot(index="Exp", columns="perturb", values="IoU_bg")
fg_tbl   = full.pivot(index="Exp", columns="perturb", values="IoU_fg")

row_order = [e[0] for e in ALL_EXPS]
col_order = ["clean","blur","brightness","gauss","hflip","rotation"]
miou_tbl = miou_tbl.loc[row_order, col_order]
bg_tbl   = bg_tbl.loc[row_order, col_order]
fg_tbl   = fg_tbl.loc[row_order, col_order]

def round3(df): return df.applymap(lambda v: float(f"{v:.3f}"))
miou_r, bg_r, fg_r = round3(miou_tbl), round3(bg_tbl), round3(fg_tbl)

print("\n=== mIoU summary (ViT attention rollout) ==="); display(miou_r)
print("\n=== IoU_bg ==="); display(bg_r)
print("\n=== IoU_fg ==="); display(fg_r)

# Save combined CSVs
Path(CFG.OUT_ROOT).mkdir(parents=True, exist_ok=True)
miou_tbl.to_csv(Path(CFG.OUT_ROOT)/"combined_mIoU_table.csv")
bg_tbl.to_csv(Path(CFG.OUT_ROOT)/"combined_IoU_bg_table.csv")
fg_tbl.to_csv(Path(CFG.OUT_ROOT)/"combined_IoU_fg_table.csv")

# LaTeX with arrows vs baseline (G_vitroll)
def latex_with_arrows(df, caption, label, base_row="G_vitroll"):
    base = df.loc[base_row]
    arrows = df.copy()
    for r in arrows.index:
        for c in arrows.columns:
            if r == base_row:
                arrows.loc[r,c] = f"{df.loc[r,c]:.3f}"
            else:
                delta = df.loc[r,c] - base[c]
                sym = "∼"
                if   delta > +1e-3: sym = "↑"
                elif delta < -1e-3: sym = "↓"
                arrows.loc[r,c] = f"{df.loc[r,c]:.3f} ({sym})"
    header = " & ".join(["Exp"] + [h.capitalize() for h in df.columns])
    lines  = [header + " \\\\ \\midrule"]
    for r in arrows.index:
        lines.append(" & ".join([r] + [arrows.loc[r,c] for c in df.columns]) + " \\\\")
    body = "\n".join(lines)
    tex = (
        "\\begin{table}[h!]\n\\centering\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        "\\begin{tabular}{lcccccc}\n\\toprule\n"
        + body +
        "\n\\bottomrule\n\\end{tabular}\n\\end{table}"
    )
    return tex

tex_miou = latex_with_arrows(miou_r, "mIoU comparison across perturbations (ViT attention rollout experiments)", "tab:vitroll-miou")
tex_bg   = latex_with_arrows(bg_r,   "Background IoU (IoU\\_bg) across perturbations (ViT attention rollout experiments)", "tab:vitroll-bg")
tex_fg   = latex_with_arrows(fg_r,   "Foreground IoU (IoU\\_fg) across perturbations (ViT attention rollout experiments)", "tab:vitroll-fg")

print(tex_miou, "\n")
print(tex_bg, "\n")
print(tex_fg)

Path(CFG.OUT_ROOT, "latex").mkdir(parents=True, exist_ok=True)
(Path(CFG.OUT_ROOT)/"latex/table_miou.tex").write_text(tex_miou)
(Path(CFG.OUT_ROOT)/"latex/table_ioubg.tex").write_text(tex_bg)
(Path(CFG.OUT_ROOT)/"latex/table_ioufg.tex").write_text(tex_fg)
print("\nSaved combined CSVs and LaTeX tables under:", CFG.OUT_ROOT)

L_vitroll_erode3:clean:   0%|          | 0/1449 [00:00<?, ?it/s]

L_vitroll_erode3:blur:   0%|          | 0/1449 [00:00<?, ?it/s]

L_vitroll_erode3:brightness:   0%|          | 0/1449 [00:00<?, ?it/s]

L_vitroll_erode3:gauss:   0%|          | 0/1449 [00:00<?, ?it/s]

L_vitroll_erode3:hflip:   0%|          | 0/1449 [00:00<?, ?it/s]

...


=== IoU_bg ===


perturb,clean,blur,brightness,gauss,hflip,rotation
Exp,,,,,,
L_vitroll_erode3,0.704,0.703,0.705,0.703,0.704,0.702



=== IoU_fg ===


perturb,clean,blur,brightness,gauss,hflip,rotation
Exp,,,,,,
L_vitroll_erode3,0.105,0.099,0.112,0.098,0.102,0.094


KeyError: 'G_vitroll'